# LightGBM-Huber on R1000/R2000 Daily Holdings

## Goal

This notebook is a **standalone experiment** for
`manager_holdings/R1000_R2000_daily_turnover21D.parquet`. It does not import the
StockRankLM project. It trains a robust LightGBM return regressor and evaluates
daily cross-sectional RankIC plus equal-weight Top-10, Bottom-10, and
Top-minus-Bottom returns.

Run it from the directory containing the `manager_holdings/` folder. If a
dependency is missing, run this once in a cell and restart the kernel:

```python
%pip install -q "lightgbm>=4.0" "scikit-learn>=1.3" "pandas>=2.0" "pyarrow>=12" "numpy>=1.24" "matplotlib>=3.7"
```

The notebook intentionally applies **no volume, turnover, price, or size
screen**. Zero-volume and low-volume securities stay in the daily universe.

## Context & Methods

### Key assumptions

1. At row date `day=t`, `TRET_T1D` is already the return from `t` to the next
   trading day. The notebook does **not** shift it again.
2. `TRET_T1D` is stored in decimal units by default (`0.01 = 1%`). Set
   `RETURN_MULTIPLIER=0.01` if the source uses percentage points.
3. The fixed comparison split is training through 2022, validation in 2023,
   and testing from 2024 onward.
4. The three primary features are trailing characteristics known at `day=t`:
   log average dollar volume, log market capitalization, and 21-day turnover.
5. This is a **three-feature liquidity/size benchmark**, not the original
   eleven-feature CRSP LightGBM-Huber model.

## Setup

In [ ]:
from __future__ import annotations

import json
import math
import os
import platform
from pathlib import Path
import warnings

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)


def env_bool(name: str, default: bool) -> bool:
    value = os.environ.get(name)
    if value is None:
        return default
    normalized = value.strip().lower()
    if normalized in {"1", "true", "yes", "y", "on"}:
        return True
    if normalized in {"0", "false", "no", "n", "off"}:
        return False
    raise ValueError(f"{name} must be a boolean-like value, got {value!r}.")


def optional_timestamp(name: str, default: str | None) -> pd.Timestamp | None:
    value = os.environ.get(name, default)
    if value is None or not str(value).strip():
        return None
    return pd.Timestamp(value)


# Paths are relative to the notebook's working directory unless overridden.
DATA_PATH = Path(
    os.environ.get(
        "HOLDINGS_DATA_PATH",
        "manager_holdings/R1000_R2000_daily_turnover21D.parquet",
    )
).expanduser()
OUTPUT_DIR = Path(
    os.environ.get(
        "HOLDINGS_OUTPUT_DIR", "lightgbm_huber_R1000_R2000_results"
    )
).expanduser()

# Fixed comparison split; each boundary can be overridden through the environment.
TRAIN_END = optional_timestamp("TRAIN_END", "2022-12-31")
VALIDATION_START = optional_timestamp("VALIDATION_START", "2023-01-01")
VALIDATION_END = optional_timestamp("VALIDATION_END", "2023-12-31")
TEST_START = optional_timestamp("TEST_START", "2024-01-01")
TEST_END = optional_timestamp("TEST_END", None)

RETURN_MULTIPLIER = float(os.environ.get("RETURN_MULTIPLIER", "1.0"))
TOP_K = int(os.environ.get("TOP_K", "10"))
SEED = int(os.environ.get("SEED", "1337"))
N_ESTIMATORS = int(os.environ.get("LGBM_N_ESTIMATORS", "500"))
EARLY_STOPPING_ROUNDS = int(
    os.environ.get("LGBM_EARLY_STOPPING_ROUNDS", "30")
)
N_JOBS = int(os.environ.get("LGBM_N_JOBS", "-1"))
RUN_ABLATIONS = env_bool("RUN_ABLATIONS", True)
SAVE_MODEL = env_bool("SAVE_MODEL", False)
SAVE_ROW_LEVEL_PREDICTIONS = env_bool("SAVE_ROW_LEVEL_PREDICTIONS", False)

if not (TRAIN_END < VALIDATION_START <= VALIDATION_END < TEST_START):
    raise ValueError("Train, validation, and test dates must be chronological and disjoint.")
if TEST_END is not None and TEST_START > TEST_END:
    raise ValueError("TEST_END must be on or after TEST_START.")
if TOP_K < 1 or N_ESTIMATORS < 1 or EARLY_STOPPING_ROUNDS < 1:
    raise ValueError("TOP_K, N_ESTIMATORS, and EARLY_STOPPING_ROUNDS must be positive.")
if not math.isfinite(RETURN_MULTIPLIER) or RETURN_MULTIPLIER == 0:
    raise ValueError("RETURN_MULTIPLIER must be finite and nonzero.")

CONFIG_VIEW = pd.Series(
    {
        "data_path": str(DATA_PATH),
        "output_dir": str(OUTPUT_DIR),
        "train_end": TRAIN_END.date().isoformat(),
        "validation": f"{VALIDATION_START.date()} to {VALIDATION_END.date()}",
        "test": f"{TEST_START.date()} to {TEST_END.date() if TEST_END is not None else 'latest'}",
        "return_multiplier": RETURN_MULTIPLIER,
        "top_k": TOP_K,
        "run_ablations": RUN_ABLATIONS,
        "save_model": SAVE_MODEL,
        "save_row_predictions": SAVE_ROW_LEVEL_PREDICTIONS,
    },
    name="value",
).to_frame()
display(CONFIG_VIEW)

## Data

### 1. Load and validate the six-column input

Only missing/non-finite targets are excluded from supervised fitting and
evaluation. Missing feature values stay in the sample because LightGBM handles
them natively. Invalid negative liquidity characteristics become missing rather
than causing an entire row to be removed. Exact zero values remain zero.

In [ ]:
REQUIRED_COLUMNS = [
    "DOLLARVOLUME_AVG21D",
    "day",
    "security",
    "TRET_T1D",
    "market_cap",
    "turnover_21day",
]

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Input not found: {DATA_PATH.resolve()}\n"
        "Place this notebook beside the manager_holdings/ directory, or set "
        "the HOLDINGS_DATA_PATH environment variable."
    )

frame = pd.read_parquet(DATA_PATH, columns=REQUIRED_COLUMNS)
frame = frame.copy()
frame["day"] = pd.to_datetime(frame["day"], errors="coerce").dt.normalize()
if frame["day"].isna().any():
    raise ValueError(f"day contains {int(frame['day'].isna().sum()):,} unparseable values.")
if frame["security"].isna().any():
    raise ValueError(f"security contains {int(frame['security'].isna().sum()):,} missing values.")
if frame.duplicated(["day", "security"]).any():
    sample = frame.loc[
        frame.duplicated(["day", "security"], keep=False), ["day", "security"]
    ].head(10)
    raise ValueError(
        "Expected at most one row per (day, security). Duplicate sample:\n"
        + sample.to_string(index=False)
    )

NUMERIC_COLUMNS = [
    "DOLLARVOLUME_AVG21D",
    "TRET_T1D",
    "market_cap",
    "turnover_21day",
]
for column in NUMERIC_COLUMNS:
    frame[column] = pd.to_numeric(frame[column], errors="coerce")
    frame.loc[~np.isfinite(frame[column]), column] = np.nan
frame["TRET_T1D"] = frame["TRET_T1D"] * RETURN_MULTIPLIER

invalid_negative_dollar_volume = int((frame["DOLLARVOLUME_AVG21D"] < 0).sum())
invalid_negative_turnover = int((frame["turnover_21day"] < 0).sum())
invalid_nonpositive_market_cap = int((frame["market_cap"] <= 0).sum())
if invalid_negative_dollar_volume:
    warnings.warn(
        f"Treating {invalid_negative_dollar_volume:,} negative dollar-volume values as missing."
    )
if invalid_negative_turnover:
    warnings.warn(
        f"Treating {invalid_negative_turnover:,} negative turnover values as missing."
    )
if invalid_nonpositive_market_cap:
    warnings.warn(
        f"Treating {invalid_nonpositive_market_cap:,} nonpositive market-cap values as missing."
    )
frame.loc[frame["DOLLARVOLUME_AVG21D"] < 0, "DOLLARVOLUME_AVG21D"] = np.nan
frame.loc[frame["turnover_21day"] < 0, "turnover_21day"] = np.nan
frame.loc[frame["market_cap"] <= 0, "market_cap"] = np.nan

frame = frame.sort_values(["day", "security"], kind="stable").reset_index(drop=True)
target_abs = frame["TRET_T1D"].abs().dropna()
quality = pd.Series(
    {
        "rows": len(frame),
        "market_dates": frame["day"].nunique(),
        "securities": frame["security"].nunique(),
        "first_day": frame["day"].min().date().isoformat(),
        "last_day": frame["day"].max().date().isoformat(),
        "missing_target_rows": int(frame["TRET_T1D"].isna().sum()),
        "zero_dollar_volume_rows_kept": int(frame["DOLLARVOLUME_AVG21D"].eq(0).sum()),
        "zero_turnover_rows_kept": int(frame["turnover_21day"].eq(0).sum()),
        "missing_dollar_volume_rows": int(frame["DOLLARVOLUME_AVG21D"].isna().sum()),
        "missing_market_cap_rows": int(frame["market_cap"].isna().sum()),
        "missing_turnover_rows": int(frame["turnover_21day"].isna().sum()),
        "median_abs_target": float(target_abs.median()) if len(target_abs) else np.nan,
        "target_abs_p99": float(target_abs.quantile(0.99)) if len(target_abs) else np.nan,
    },
    name="value",
).to_frame()
display(quality)
display(frame.head(5))

if target_abs.empty:
    raise ValueError("TRET_T1D has no finite observations.")
if target_abs.median() > 0.20:
    warnings.warn(
        "Median absolute TRET_T1D exceeds 20%. Confirm its units; if values are "
        "percentage points, rerun with RETURN_MULTIPLIER=0.01."
    )

### 2. Construct point-in-time features and fixed date splits

In [ ]:
frame["log_dollar_volume_21d"] = np.log1p(frame["DOLLARVOLUME_AVG21D"])
frame["log_market_cap"] = np.log(frame["market_cap"])
frame["turnover_21day"] = frame["turnover_21day"].astype(float)

PRIMARY_FEATURES = [
    "log_dollar_volume_21d",
    "log_market_cap",
    "turnover_21day",
]
FEATURE_SETS = {"all_three": PRIMARY_FEATURES}
if RUN_ABLATIONS:
    FEATURE_SETS.update(
        {
            "turnover_only": ["turnover_21day"],
            "dollar_volume_only": ["log_dollar_volume_21d"],
            "market_cap_only": ["log_market_cap"],
            "no_turnover": ["log_dollar_volume_21d", "log_market_cap"],
        }
    )

finite_target = frame["TRET_T1D"].notna()
train_mask = finite_target & (frame["day"] <= TRAIN_END)
validation_mask = (
    finite_target
    & (frame["day"] >= VALIDATION_START)
    & (frame["day"] <= VALIDATION_END)
)
test_mask = finite_target & (frame["day"] >= TEST_START)
if TEST_END is not None:
    test_mask &= frame["day"] <= TEST_END

train = frame.loc[train_mask].copy()
validation = frame.loc[validation_mask].copy()
test = frame.loc[test_mask].copy()
for name, split in {"train": train, "validation": validation, "test": test}.items():
    if split.empty:
        raise ValueError(
            f"{name} split is empty under the configured dates. "
            "Edit the date parameters near the top of the notebook."
        )
    if split["day"].nunique() < 2:
        raise ValueError(f"{name} split must contain at least two market dates.")

minimum_test_assets = int(test.groupby("day", sort=True).size().min())
if minimum_test_assets < TOP_K:
    raise ValueError(
        f"At least one test date has only {minimum_test_assets} labeled assets, "
        f"fewer than TOP_K={TOP_K}."
    )

split_table = pd.DataFrame(
    [
        {
            "split": name,
            "start": split["day"].min().date().isoformat(),
            "end": split["day"].max().date().isoformat(),
            "dates": split["day"].nunique(),
            "rows": len(split),
            "securities": split["security"].nunique(),
            "minimum_daily_assets": split.groupby("day").size().min(),
        }
        for name, split in (
            ("train", train),
            ("validation", validation),
            ("test", test),
        )
    ]
)
display(split_table)
display(pd.Series({name: " | ".join(columns) for name, columns in FEATURE_SETS.items()}, name="features").to_frame())

## Results

### 3. Define the model and evaluation functions

In [ ]:
def fit_huber(feature_columns: list[str]) -> lgb.LGBMRegressor:
    min_child_samples = min(200, max(1, len(train) // 100))
    model = lgb.LGBMRegressor(
        objective="huber",
        alpha=0.9,
        n_estimators=N_ESTIMATORS,
        learning_rate=0.05,
        num_leaves=63,
        min_child_samples=min_child_samples,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.9,
        reg_lambda=1.0,
        max_bin=127,
        random_state=SEED,
        n_jobs=N_JOBS,
        verbosity=-1,
        deterministic=True,
        force_col_wise=True,
    )
    model.fit(
        train[feature_columns].astype("float32"),
        train["TRET_T1D"].astype("float32"),
        eval_set=[
            (
                validation[feature_columns].astype("float32"),
                validation["TRET_T1D"].astype("float32"),
            )
        ],
        eval_metric="l2",
        callbacks=[
            lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False),
            lgb.log_evaluation(period=50),
        ],
    )
    return model


def hac_mean_t_stat(values: pd.Series, max_lag: int = 5) -> float:
    # Newey-West t-statistic for a daily mean with Bartlett weights.
    array = pd.to_numeric(values, errors="coerce").to_numpy(dtype=float)
    array = array[np.isfinite(array)]
    count = len(array)
    if count < 2:
        return math.nan
    centered = array - array.mean()
    long_run_variance = float(centered @ centered / count)
    for lag in range(1, min(max_lag, count - 1) + 1):
        autocovariance = float(centered[lag:] @ centered[:-lag] / count)
        weight = 1.0 - lag / (max_lag + 1.0)
        long_run_variance += 2.0 * weight * autocovariance
    if not math.isfinite(long_run_variance) or long_run_variance <= 0:
        return math.nan
    standard_error = math.sqrt(long_run_variance / count)
    return float(array.mean() / standard_error)


def max_drawdown(returns: pd.Series) -> float:
    daily = pd.to_numeric(returns, errors="coerce")
    daily = daily[np.isfinite(daily)]
    if daily.empty or (daily < -1.0).any():
        return math.nan
    equity = (1.0 + daily).cumprod()
    running_peak = np.maximum.accumulate(np.r_[1.0, equity.to_numpy()])[1:]
    return float(np.min(equity.to_numpy() / running_peak - 1.0))


def return_summary(values: pd.Series) -> dict[str, float | int]:
    daily = pd.to_numeric(values, errors="coerce")
    daily = daily[np.isfinite(daily)]
    observations = len(daily)
    if observations == 0:
        return {
            "observations": 0,
            "annualized_mean": math.nan,
            "annualized_volatility": math.nan,
            "sharpe": math.nan,
            "max_drawdown": math.nan,
        }
    annualized_mean = float(daily.mean() * 252.0)
    annualized_volatility = (
        float(daily.std(ddof=1) * math.sqrt(252.0))
        if observations > 1
        else math.nan
    )
    sharpe = (
        annualized_mean / annualized_volatility
        if annualized_volatility > 0 and math.isfinite(annualized_volatility)
        else math.nan
    )
    return {
        "observations": observations,
        "annualized_mean": annualized_mean,
        "annualized_volatility": annualized_volatility,
        "sharpe": sharpe,
        "max_drawdown": max_drawdown(daily),
    }


def score_test(model: lgb.LGBMRegressor, feature_columns: list[str]) -> pd.DataFrame:
    predictions = test[
        [
            "day",
            "security",
            "TRET_T1D",
            "DOLLARVOLUME_AVG21D",
            "market_cap",
            "turnover_21day",
        ]
    ].copy()
    best_iteration = int(model.best_iteration_ or model.n_estimators)
    predictions["score"] = model.predict(
        test[feature_columns].astype("float32"), num_iteration=best_iteration
    )
    if not np.isfinite(predictions["score"]).all():
        raise ValueError("LightGBM produced a non-finite test score.")
    return predictions


def daily_rank_and_portfolios(predictions: pd.DataFrame) -> pd.DataFrame:
    records: list[dict[str, float | int | pd.Timestamp]] = []
    for day, group in predictions.groupby("day", sort=True):
        ranked = group.sort_values(
            ["score", "security"], ascending=[False, True], kind="stable"
        )
        limit = min(TOP_K, len(ranked))
        top = ranked.head(limit)
        bottom = ranked.tail(limit)
        score_rank = ranked["score"].rank(method="average")
        return_rank = ranked["TRET_T1D"].rank(method="average")
        rank_ic = score_rank.corr(return_rank)
        top_return = float(top["TRET_T1D"].mean())
        bottom_return = float(bottom["TRET_T1D"].mean())
        records.append(
            {
                "day": day,
                "rank_ic": float(rank_ic) if pd.notna(rank_ic) else math.nan,
                "top_return": top_return,
                "bottom_return": bottom_return,
                "spread_return": top_return - bottom_return,
                "top_holdings": len(top),
                "bottom_holdings": len(bottom),
                "eligible_assets": len(ranked),
            }
        )
    return pd.DataFrame.from_records(records)


def model_summary(
    model_name: str,
    feature_columns: list[str],
    model: lgb.LGBMRegressor,
    daily: pd.DataFrame,
) -> dict[str, float | int | str]:
    top = return_summary(daily["top_return"])
    bottom = return_summary(daily["bottom_return"])
    spread = return_summary(daily["spread_return"])
    return {
        "model": model_name,
        "features": " | ".join(feature_columns),
        "best_iteration": int(model.best_iteration_ or model.n_estimators),
        "train_rows": len(train),
        "validation_rows": len(validation),
        "test_rows": len(test),
        "test_dates": int(daily["day"].nunique()),
        "rank_ic": float(daily["rank_ic"].mean()),
        "top_annualized_mean": top["annualized_mean"],
        "top_sharpe": top["sharpe"],
        "bottom_annualized_mean": bottom["annualized_mean"],
        "bottom_sharpe": bottom["sharpe"],
        "spread_annualized_mean": spread["annualized_mean"],
        "spread_annualized_volatility": spread["annualized_volatility"],
        "spread_sharpe": spread["sharpe"],
        "spread_max_drawdown": spread["max_drawdown"],
        "spread_hac5_t": hac_mean_t_stat(daily["spread_return"], max_lag=5),
    }


def annual_rows(model_name: str, daily: pd.DataFrame) -> list[dict[str, float | int | str]]:
    rows: list[dict[str, float | int | str]] = []
    for year, group in daily.groupby(daily["day"].dt.year, sort=True):
        top = return_summary(group["top_return"])
        bottom = return_summary(group["bottom_return"])
        spread = return_summary(group["spread_return"])
        rows.append(
            {
                "model": model_name,
                "period": str(int(year)),
                "start": group["day"].min().date().isoformat(),
                "end": group["day"].max().date().isoformat(),
                "observations": len(group),
                "rank_ic": float(group["rank_ic"].mean()),
                "top_annualized_mean": top["annualized_mean"],
                "top_sharpe": top["sharpe"],
                "bottom_annualized_mean": bottom["annualized_mean"],
                "bottom_sharpe": bottom["sharpe"],
                "spread_annualized_mean": spread["annualized_mean"],
                "spread_sharpe": spread["sharpe"],
                "spread_hac5_t": hac_mean_t_stat(group["spread_return"], max_lag=5),
            }
        )
    return rows


def selection_characteristic_rows(
    model_name: str, predictions: pd.DataFrame
) -> list[dict[str, float | int | str]]:
    daily_rows: list[dict[str, float | int | str]] = []
    for day, group in predictions.groupby("day", sort=True):
        ranked = group.sort_values(
            ["score", "security"], ascending=[False, True], kind="stable"
        )
        limit = min(TOP_K, len(ranked))
        selections = {
            "top": ranked.head(limit),
            "universe": ranked,
            "bottom": ranked.tail(limit),
        }
        for leg, selected in selections.items():
            daily_rows.append(
                {
                    "model": model_name,
                    "day": day,
                    "leg": leg,
                    "holdings": len(selected),
                    "median_dollar_volume_21d": float(selected["DOLLARVOLUME_AVG21D"].median()),
                    "median_market_cap": float(selected["market_cap"].median()),
                    "median_turnover_21day": float(selected["turnover_21day"].median()),
                    "zero_dollar_volume_share": float(selected["DOLLARVOLUME_AVG21D"].eq(0).mean()),
                }
            )
    daily_frame = pd.DataFrame.from_records(daily_rows)
    result = (
        daily_frame.groupby(["model", "leg"], sort=True, as_index=False)
        .agg(
            dates=("day", "nunique"),
            mean_holdings=("holdings", "mean"),
            mean_daily_median_dollar_volume_21d=("median_dollar_volume_21d", "mean"),
            mean_daily_median_market_cap=("median_market_cap", "mean"),
            mean_daily_median_turnover_21day=("median_turnover_21day", "mean"),
            mean_zero_dollar_volume_share=("zero_dollar_volume_share", "mean"),
        )
    )
    return result.to_dict(orient="records")

### 4. Train the primary model and optional feature ablations

In [ ]:
models: dict[str, lgb.LGBMRegressor] = {}
predictions_by_model: dict[str, pd.DataFrame] = {}
daily_by_model: dict[str, pd.DataFrame] = {}
summary_rows: list[dict[str, float | int | str]] = []
year_rows: list[dict[str, float | int | str]] = []
importance_rows: list[dict[str, float | int | str]] = []
characteristic_rows: list[dict[str, float | int | str]] = []

for model_name, feature_columns in FEATURE_SETS.items():
    print(f"Training {model_name}: {feature_columns}")
    fitted = fit_huber(feature_columns)
    scored = score_test(fitted, feature_columns)
    daily = daily_rank_and_portfolios(scored)
    models[model_name] = fitted
    predictions_by_model[model_name] = scored
    daily_by_model[model_name] = daily
    summary_rows.append(model_summary(model_name, feature_columns, fitted, daily))
    year_rows.extend(annual_rows(model_name, daily))
    characteristic_rows.extend(selection_characteristic_rows(model_name, scored))

    gain = fitted.booster_.feature_importance(importance_type="gain")
    gain_total = float(gain.sum())
    for feature, value in zip(feature_columns, gain):
        importance_rows.append(
            {
                "model": model_name,
                "feature": feature,
                "gain": float(value),
                "gain_share": float(value / gain_total) if gain_total > 0 else math.nan,
            }
        )

summary_table = pd.DataFrame.from_records(summary_rows)
year_table = pd.DataFrame.from_records(year_rows)
feature_importance = pd.DataFrame.from_records(importance_rows).sort_values(
    ["model", "gain"], ascending=[True, False], kind="stable"
)
selection_characteristics = pd.DataFrame.from_records(characteristic_rows)

display(summary_table.round(6))
display(year_table.round(6))
display(feature_importance.round(6))
display(selection_characteristics.round(6))

### 5. Save compact, reproducible outputs

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
summary_table.to_csv(OUTPUT_DIR / "summary.csv", index=False)
year_table.to_csv(OUTPUT_DIR / "year_table.csv", index=False)
feature_importance.to_csv(OUTPUT_DIR / "feature_importance.csv", index=False)
selection_characteristics.to_csv(
    OUTPUT_DIR / "selection_characteristics.csv", index=False
)

primary_daily = daily_by_model["all_three"].copy()
primary_daily.to_parquet(OUTPUT_DIR / "daily_portfolios.parquet", index=False)

if SAVE_MODEL:
    for model_name, fitted in models.items():
        fitted.booster_.save_model(
            str(OUTPUT_DIR / f"{model_name}_model.txt"),
            num_iteration=int(fitted.best_iteration_ or fitted.n_estimators),
        )

if SAVE_ROW_LEVEL_PREDICTIONS:
    pd.concat(
        [values.assign(model=name) for name, values in predictions_by_model.items()],
        ignore_index=True,
    ).to_parquet(OUTPUT_DIR / "row_level_predictions.parquet", index=False)

run_config = {
    "data_path": str(DATA_PATH.resolve()),
    "data_rows": len(frame),
    "data_first_day": frame["day"].min().date().isoformat(),
    "data_last_day": frame["day"].max().date().isoformat(),
    "train_end": TRAIN_END.date().isoformat(),
    "validation_start": VALIDATION_START.date().isoformat(),
    "validation_end": VALIDATION_END.date().isoformat(),
    "test_start": TEST_START.date().isoformat(),
    "test_end": TEST_END.date().isoformat() if TEST_END is not None else None,
    "return_multiplier": RETURN_MULTIPLIER,
    "top_k": TOP_K,
    "seed": SEED,
    "n_estimators": N_ESTIMATORS,
    "early_stopping_rounds": EARLY_STOPPING_ROUNDS,
    "run_ablations": RUN_ABLATIONS,
    "save_model": SAVE_MODEL,
    "save_row_level_predictions": SAVE_ROW_LEVEL_PREDICTIONS,
    "feature_sets": FEATURE_SETS,
    "zero_dollar_volume_rows_kept": int(frame["DOLLARVOLUME_AVG21D"].eq(0).sum()),
    "python_version": platform.python_version(),
    "pandas_version": pd.__version__,
    "numpy_version": np.__version__,
    "lightgbm_version": lgb.__version__,
}
(OUTPUT_DIR / "run_config.json").write_text(
    json.dumps(run_config, indent=2, sort_keys=True), encoding="utf-8"
)

saved_files = pd.DataFrame(
    [
        {"file": path.name, "bytes": path.stat().st_size}
        for path in sorted(OUTPUT_DIR.iterdir())
        if path.is_file()
    ]
)
display(saved_files)
print(f"Saved compact results to: {OUTPUT_DIR.resolve()}")

### 6. Inspect Top, Bottom, and Top-minus-Bottom separately

In [ ]:
plot_frame = primary_daily.sort_values("day").copy()
plot_frame["cumulative_top_arithmetic"] = plot_frame["top_return"].cumsum()
plot_frame["cumulative_bottom_arithmetic"] = plot_frame["bottom_return"].cumsum()
plot_frame["cumulative_spread_arithmetic"] = plot_frame["spread_return"].cumsum()

fig, ax = plt.subplots(figsize=(11, 5.5))
ax.plot(
    plot_frame["day"],
    100 * plot_frame["cumulative_top_arithmetic"],
    label="Top-10",
    linewidth=1.8,
)
ax.plot(
    plot_frame["day"],
    100 * plot_frame["cumulative_bottom_arithmetic"],
    label="Bottom-10 underlying return",
    linewidth=1.8,
)
ax.plot(
    plot_frame["day"],
    100 * plot_frame["cumulative_spread_arithmetic"],
    label="Top minus Bottom",
    linewidth=2.3,
)
ax.axhline(0, color="black", linewidth=0.8, alpha=0.6)
ax.set_title("LightGBM-Huber: cumulative arithmetic daily returns")
ax.set_ylabel("Cumulative return (percentage points)")
ax.set_xlabel("Test date")
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
plt.show()

## Takeaways

Read the executed output in this order:

1. **RankIC**: whether the score orders the full daily cross-section correctly
   on average.
2. **Top and Bottom separately**: whether the spread comes from finding winners,
   losers, or both. A large spread with little Top improvement can still be a
   useful ranking result, but it is economically a Bottom-selection effect.
3. **HAC(5) t-statistic**: whether the average daily spread is large relative to
   short-horizon dependence in its time series.
4. **Ablations**: whether the signal is turnover, dollar-volume, size, or an
   interaction learned by the tree.
5. **Selection characteristics**: whether the model systematically pushes the
   Top or Bottom leg into smaller or less-liquid securities.

The notebook reports gross predictive and portfolio evidence. It deliberately
does not turn this into an execution-cost study, and it does not claim direct
comparability with the prior eleven-feature CRSP baseline.